# BrewStock Demand Forecast Starter
Notebook ini untuk eksplorasi demand harian berbasis dataset Kaggle Maven Roasters.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# Dibantu AI: resolveProjectRoot
def resolveProjectRoot() -> Path:
    currentPath = Path.cwd().resolve()
    for parentPath in [currentPath] + list(currentPath.parents):
        if (parentPath / 'ml').exists() and (parentPath / 'backend').exists():
            return parentPath
    raise ValueError('Project root not found')

In [ ]:
projectRoot = resolveProjectRoot()
rawDataPath = projectRoot / 'ml/forecast/data/raw/trendsProductCoffeeShopSalesRevenueDataset/coffee-shop-sales-revenue.csv'
processedDirPath = projectRoot / 'ml/forecast/data/processed'
processedDirPath.mkdir(parents=True, exist_ok=True)
dailyDemandPath = processedDirPath / 'dailyDemandBySku.csv'
rawDataPath

In [ ]:
# Dibantu AI: loadSalesData
def loadSalesData(rawDataPath: Path) -> pd.DataFrame:
    frameValue = pd.read_csv(rawDataPath)
    frameValue['transaction_date'] = pd.to_datetime(frameValue['transaction_date'], format='%Y-%m-%d')
    frameValue['transaction_time'] = pd.to_datetime(frameValue['transaction_time'], format='%H:%M:%S').dt.time
    return frameValue

In [ ]:
# Dibantu AI: buildDailySkuDemand
def buildDailySkuDemand(salesFrame: pd.DataFrame) -> pd.DataFrame:
    frameValue = salesFrame.copy()
    frameValue['skuId'] = (
        frameValue['product_id'].astype(str) + '-' + frameValue['store_id'].astype(str)
    )
    groupedValue = (
        frameValue
        .groupby(['transaction_date', 'skuId'], as_index=False)['transaction_qty']
        .sum()
        .rename(columns={'transaction_date': 'transactionDate', 'transaction_qty': 'demandQuantity'})
        .sort_values(['transactionDate', 'skuId'])
    )
    return groupedValue

In [ ]:
salesFrame = loadSalesData(rawDataPath)
dailyDemandFrame = buildDailySkuDemand(salesFrame)
dailyDemandFrame.to_csv(dailyDemandPath, index=False)
dailyDemandFrame.head()

In [ ]:
# Dibantu AI: buildProphetFrame
def buildProphetFrame(dailyDemandFrame: pd.DataFrame, skuId: str) -> pd.DataFrame:
    filteredFrame = dailyDemandFrame[dailyDemandFrame['skuId'] == skuId].copy()
    filteredFrame = filteredFrame.rename(columns={'transactionDate': 'ds', 'demandQuantity': 'y'})
    filteredFrame = filteredFrame.sort_values('ds')
    return filteredFrame[['ds', 'y']]

In [ ]:
topSkuSeries = (
    dailyDemandFrame.groupby('skuId', as_index=False)['demandQuantity']
    .sum()
    .sort_values('demandQuantity', ascending=False)
)
topSkuId = topSkuSeries.iloc[0]['skuId']
prophetFrame = buildProphetFrame(dailyDemandFrame, topSkuId)
topSkuId, prophetFrame.head()

In [ ]:
# Dibantu AI: runNaiveForecast
def runNaiveForecast(prophetFrame: pd.DataFrame, horizonDays: int = 14) -> pd.DataFrame:
    if prophetFrame.empty:
        raise ValueError('No series data')
    lastDate = pd.to_datetime(prophetFrame['ds'].max())
    lastValue = float(prophetFrame['y'].iloc[-1])
    forecastDateSeries = pd.date_range(lastDate + pd.Timedelta(days=1), periods=horizonDays, freq='D')
    forecastFrame = pd.DataFrame({'transactionDate': forecastDateSeries, 'forecastQuantity': np.repeat(lastValue, horizonDays)})
    return forecastFrame

In [ ]:
naiveForecastFrame = runNaiveForecast(prophetFrame, horizonDays=14)
naiveForecastFrame.head()

## Next Step
- Ganti baseline ke Prophet/XGBoost.
- Kirim `dailyDemandBySku.csv` ke endpoint forecast backend.